# EODAG / Data Gateway Building Block

New features introduced in EODAG [v4.5.0](https://github.com/CS-SI/eodag/releases/tag/v4.5.0) for EOEPCA:

- [Provider configuration simplification and optimisation](#Provider-configuration-simplification-and-optimisation)
- [OGC API Records search plugin](#OGC-API-Records-search-plugin)

- - -

# Provider configuration simplification and optimisation

Until EODAG v4.4.0:

- Single multi-document YAML file ([eodag/resources/providers.yml](https://github.com/CS-SI/eodag/blob/v4.4.0/eodag/resources/providers.yml)) shipped with EODAG package.
  
  &rarr; More than 7000 lines: difficult to maintain and to discover for new users / contributors.
  ```yaml
  ---
  !provider
    name: usgs
    priority: 0
    [...]
    api: !plugin
      type: UsgsApi
    [...]
  ---
  !provider
    name: creodias
    [...]
  ```

- Custom YAML tags `!provider`, `!plugin` and and Python types `!!python/tuple`

  - PyYAML-specific with limited loaders compatility
  - Automates `ProviderConfig` and `PluginConfig` objects creation **BUT** adds complexity to YAML configuration
  - Provider config can be overridden by [user configuration file](https://github.com/CS-SI/eodag/blob/v4.4.0/eodag/resources/user_conf_template.yml) which uses a **DIFFERENT** syntax without YAML tags.

Starting v4.5.0:

- Provider conf splitted by provider ([#2228](https://github.com/CS-SI/eodag/pull/2228))
```text
eodag/resources/providers
├── aws_eos.yml
├── cop_ads.yml
├── cop_cds.yml
├── cop_dataspace_s3.yml
├── cop_dataspace.yml
├── cop_ewds.yml
├── cop_ghsl.yml
├── cop_marine.yml
├── creodias_s3.yml
├── creodias.yml
├── dedl.yml
├── dedt_lumi.yml
├── dedt_mn5.yml
├── dlr_eoc_geoservice.yml
├── earth_search_gcs.yml
├── earth_search.yml
├── ecmwf.yml
├── eocat.yml
├── eumetsat_ds.yml
├── fedeo_ceda.yml
├── geodes_s3.yml
├── geodes.yml
├── hydroweb_next.yml
├── meteoblue.yml
├── planetary_computer.yml
├── sara.yml
├── theia.yml
├── usgs_satapi_aws.yml
├── usgs.yml
├── wekeo_cmems.yml
├── wekeo_ecmwf.yml
└── wekeo_main.yml
```
- Removed YAML tags  and and Python types ([#2251](https://github.com/CS-SI/eodag/pull/2251))

    - Harmonized configuration syntax between core providers configuration and User configuration file
    - Enables the usage of `CSafeLoader` as PyYAML loader : `EODataAccessGateway` cold instantiation is now **~ 40% faster**.
  

- - -

# OGC API Records search plugin

In this tutorial we will show you how to use EODAG to search data from providers exposing data through [OGC API Records Core](https://docs.ogc.org/is/19-072/19-072.html) using [OARSearch](https://eodag.readthedocs.io/en/latest/plugins_reference/generated/eodag.plugins.search.oar.OARSearch.html) plugin.

In [ ]:
from importlib.metadata import version
from packaging.version import Version

assert Version(version("eodag")) >= Version("4.5.0")

In [ ]:
from eodag import EODataAccessGateway

dag = EODataAccessGateway()

## Add a new provider

Add [geomet](https://api.weather.gc.ca) as new provider using [add_provider()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.add_provider). 

GeoMet-OGC-API provides public access to the Meteorological Service of Canada (MSC) and Environment and Climate Change Canada (ECCC) data. 

Only search plugin type and `api_endpoint` are required, all remaining settings are kept as defaults.

In [ ]:
dag.add_provider(
    "geomet", 
    search={
        "type": "OARSearch", 
        "api_endpoint": "https://api.weather.gc.ca",
    },
)

## Discover collections and queryables

List available collections for this provider:

In [ ]:
dag.list_collections(provider="geomet")

List queryable parameters for `climate-daily` collection:

In [ ]:
dag.list_queryables(provider="geomet", collection="climate-daily")

## Search data

Now search for `climate-daily` data on `2025-07-17` around Montreal in southern Quebec.

In [ ]:
results = dag.search(
    provider="geomet",
    collection="climate-daily",
    start="2025-07-17", end="2025-07-17",
    geom=[-74, 45, -73, 46],
    limit=10, count=True,
)
results

Consume all pages to get all results:

In [ ]:
from collections import deque

deque(results.next_page(update=True))

print(f"Got {len(results)} results")

## Plot results on a map

Now plot on a map `TOTAL_PRECIPITATION` (mm) data from results properties.

In [ ]:
import folium

fmap = folium.Map([45.5, -73], zoom_start=9)

# Create a layer that represents the search area in red
folium.Rectangle(
    bounds=[[45, -74], [46, -73]],
    color="red",
    tooltip="Search extent"
).add_to(fmap)

folium.GeoJson(
    data=results,
    marker=folium.Circle(radius=4, fill_color="orange", fill_opacity=0.4, color="black", weight=1),
    tooltip=folium.GeoJsonTooltip(fields=["geomet:STATION_NAME", "geomet:LOCAL_DATE", "geomet:TOTAL_PRECIPITATION"]),
    style_function=lambda x: {
        "fillColor": "blue",
        "radius": (x['properties']['geomet:TOTAL_PRECIPITATION'] or 0)*500,
    },
    highlight_function=lambda x: {"fillOpacity": 0.8},
    zoom_on_click=True,
).add_to(fmap)

fmap

## Search, filter and plot results over a wider area

Search results over a wider area (whole Quebec region). We'll use queryable province code to select desired Area Of Interest.

Filter out results without valid data.

Plot results on a map.

In [ ]:
more_results = dag.search_all(
    provider="geomet",
    collection="climate-daily",
    start="2025-07-17", end="2025-07-17",
    PROVINCE_CODE="QC",
    limit=10000
)
len(more_results)

Update search plugin [pagination.max_limit](https://eodag.readthedocs.io/en/latest/plugins.html#eodag.config.PluginConfig.Pagination.max_limit) to suppress warning

In [ ]:
dag.update_providers_config(
    """
    geomet:
        search:
            pagination:
                max_limit: 10000
    """
)

In [ ]:
more_results = dag.search_all(
    provider="geomet",
    collection="climate-daily",
    start="2025-07-17", end="2025-07-17",
    PROVINCE_CODE="QC",
    limit=10000
)
len(more_results)

In [ ]:
# Filter out empty `geomet:TOTAL_PRECIPITATION`

filtered = more_results.filter_property(operator="ne", **{"geomet:TOTAL_PRECIPITATION": None})
len(filtered)

In [ ]:
fmap2 = folium.Map([55, -73], zoom_start=5, tiles="CartoDB Voyager")

folium.GeoJson(
    data=filtered,
    marker=folium.Circle(radius=4, fill_color="orange", fill_opacity=0.4, color="black", weight=1),
    tooltip=folium.GeoJsonTooltip(fields=["geomet:STATION_NAME", "geomet:LOCAL_DATE", "geomet:TOTAL_PRECIPITATION"]),
    style_function=lambda x: {
        "fillColor": "blue",
        "radius": (x['properties']['geomet:TOTAL_PRECIPITATION'] or 0)*500,
    },
    highlight_function=lambda x: {"fillOpacity": 0.8},
    zoom_on_click=True,
).add_to(fmap2)

fmap2

- - -

# Note on `eocat` provider API update

https://github.com/CS-SI/eodag/issues/2284

**ESA EOCAT** (Heritage and Third Party missions) catalogue is no more available, and is replaced with ESA MAAP.

Associated provider configuration needs be deleted / replaced.

https://earth.esa.int/eogateway/news/eocat-backend-catalogue-dismissal

> The [EOCAT Backend Catalogue](http://eocat.esa.int/eo-catalogue) will be dismissed on 10 June 2026.
The [ESA MAAP Catalogue](http://catalog.maap.eo.esa.int/catalogue/) replaces it, following the recent full migration of the content from EOCAT. The APIs for discovering and accessing data through the ESA MAAP Catalogue are mostly the same as the ones previously provided by EOCAT and are described in the [MAAP User Guides](https://portal.maap.eo.esa.int/catalogue/).
We strongly encourage you to start migrating your workflows from the EOCAT Backend Catalogue endpoint to the ESA MAAP Catalogue endpoint.
The authentication system, EO Sign In, and all the other interfaces to access ESA data remain unchanged.